# DMDU workshop: Mumbai October 2026
___

> **ADDED — not part of the original notebook.**
> This is **Part 2** of the original notebook, containing the policy analysis section. It runs on **EMA Workbench 2.5.3**, the version this section's code was written for. No original code cell has been modified.
>
> This part does **not** need any result from Part 1. It only needs the original setup cells, which are repeated below exactly as they appear in Part 1.
>
> Run the ADDED install cell below **first**, before any other cell. Use a separate session (or Colab notebook) from Part 1, because the two parts use different versions.

In [ ]:
# Installs the version this part was written for. Run this cell first, before any other cell.
%pip install -q "ema_workbench[recommended]==2.5.3"
import ema_workbench
assert ema_workbench.__version__ == "2.5.3", (
    f"Found {ema_workbench.__version__}. Restart the session (Runtime > Restart session) and run this cell first."
)
print("ADDED CHECK: EMA Workbench", ema_workbench.__version__)

Lets install the workbench.
From version 2.5.0 the workbench requires Python 3.9 or higher. Make sure to update the python kernel of this notebook to Version 3.9 before installing the workbench.


In [ ]:
try:
    import ema_workbench
    print(f"EMA Workbench {ema_workbench.__version__} is installed and ready.")
except ImportError:
    print("EMA Workbench is not installed. Run: %pip install ema_workbench")
    %pip install -U "ema_workbench[recommended]"
    import ema_workbench
    print(f"EMA Workbench {ema_workbench.__version__} is installed and ready.")

> **ADDED — not part of the original notebook.**
> The cells in this section are copied **unchanged** from Part 1. They load the flu model and define `model`, its uncertainties and its outcomes. The policy section needs these; it does not need the ensemble, the Sobol results or the clusters from Part 1.

Import all libraries

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ema_workbench.analysis import lines
from scipy.stats import gaussian_kde
from IPython.display import display
from ema_workbench import Model, RealParameter, TimeSeriesOutcome, SequentialEvaluator, perform_experiments
from ema_workbench.analysis.plotting import kde_over_time
from ema_workbench.analysis import Density
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol, pawn
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, balanced_accuracy_score, silhouette_score

Set seed and number of runs

### Choose the outcome and scalar metric
Default: **infected population in region 1**, using its **peak** for sensitivity and threshold discovery.
All ten stocks in both regions are available through OUTCOME_NAMES. Change OUTCOME, then rerun this settings cell and all subsequent cells. All plots, Sobol, PAWN, convergence, threshold discovery, early-peak discovery and clustering use that setting.
METRIC selects peak, final, or time integral for sensitivity and threshold discovery. Final-state KDE always uses the final time. Clustering uses full trajectories; early-peak discovery uses the stock maximum.
For cumulative deceased or recovered stocks, final is usually useful. Monotonic susceptible/immune trajectories may peak initially; choose final or integral if peak is uninformative. Adjust OUTCOME_THRESHOLD to match the selected metric and units.
The ensemble stores all ten stocks. The structured Sobol design is rerun for the chosen scalar metric; its outputs cannot be substituted from the exploratory sample.


In [ ]:
SEED = 42
NR_EXPERIMENTS = 1000
SOBOL_N = 256  # power of two; actual simulations = N * (17 + 2)

# Change these settings, then rerun this cell and all analysis cells below it.
OUTCOME = "infected_population_region_1"
METRIC = "peak"  # "peak", "final", or "integral"
OUTCOME_NAMES = [f"{stock}_population_region_{region}"
                 for stock in ["susceptible", "infected", "recovered", "immune", "deceased"]
                 for region in [1, 2]]
assert OUTCOME in OUTCOME_NAMES
assert METRIC in {"peak", "final", "integral"}
OUTCOME_LABEL = OUTCOME.replace("_", " ").capitalize()
METRIC_LABEL = f"{METRIC.capitalize()} of {OUTCOME_LABEL.lower()}"
DISPLAY_SCALE = 1e6
AXIS_LABEL = f"{OUTCOME_LABEL} (millions of people)"
# Teaching thresholds, not policy recommendations. Adjust to the chosen outcome/metric.
OUTCOME_THRESHOLD = 1_000_000
EARLY_TIME = 15
MIN_PEAK_POPULATION = 10_000

def summarize_trajectory(values, times):
    values = np.asarray(values, dtype=float)
    if METRIC == "peak":
        return np.max(values, axis=-1)
    if METRIC == "final":
        return values[..., -1]
    # Trapezoidal integral, compatible with NumPy versions before and after 2.0.
    return np.sum((values[..., 1:] + values[..., :-1]) * .5 * np.diff(times), axis=-1)

Check path

In [ ]:
MODEL_PATH = Path.cwd() / 'models' / 'Flu model.py'
assert MODEL_PATH.is_file(), f'Cannot find {MODEL_PATH}; change MODEL_PATH if needed.'
spec = importlib.util.spec_from_file_location('flu_module', MODEL_PATH)
flu_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = flu_module
spec.loader.exec_module(flu_module)
flu_model = flu_module.flu_model

In [ ]:
# (code, descriptive name, lower bound, upper bound)
PARAMETERS = [
    ('x11', 'Seasonal immunity R1', 0, .5),
    ('x12', 'Seasonal immunity R2', 0, .5),
    ('x21', 'Fatality rate R1', .0001, .1),
    ('x22', 'Fatality rate R2', .0001, .1),
    ('x31', 'Initial immune fraction R1', 0, .5),
    ('x32', 'Initial immune fraction R2', 0, .5),
    ('x41', 'Interregional contact rate', 0, .9),
    ('x51', 'Permanent immunity R1', 0, .5),
    ('x52', 'Permanent immunity R2', 0, .5),
    ('x61', 'Recovery time R1', .1, .8),
    ('x62', 'Recovery time R2', .1, .8),
    ('x81', 'Root contact rate R1', 1, 10),
    ('x82', 'Root contact rate R2', 1, 10),
    ('x91', 'Infection rate R1', 0, .1),
    ('x92', 'Infection rate R2', 0, .1),
    ('x101', 'Normal contact rate R1', 0, 200),
    ('x102', 'Normal contact rate R2', 0, 200),
]
names = [p[0] for p in PARAMETERS]
labels = {p[0]: f'{p[0]}: {p[1]}' for p in PARAMETERS}
problem = {'num_vars': len(names), 'names': names,
           'bounds': [[p[2], p[3]] for p in PARAMETERS]}
model = Model('mexicanFlu', function=flu_model)
model.uncertainties = [RealParameter(p[0], p[2], p[3]) for p in PARAMETERS]

Set model outcomes

In [ ]:
# Record all ten stocks so plots can switch outcomes without rerunning the ensemble.
model.outcomes = [TimeSeriesOutcome("TIME")] + [TimeSeriesOutcome(n) for n in OUTCOME_NAMES]
# Check one valid parameter combination before launching the ensemble.
probe = flu_model(**{p[0]: (p[2] + p[3]) / 2 for p in PARAMETERS})
missing = set(OUTCOME_NAMES) - set(probe)
assert not missing, f"Model is missing stock histories: {missing}. Use the supplied all-stocks model."

## 3. Policy Analysis

A **scenario** describes the external conditions. A **policy** describes the actions we choose.
An **experiment** combines one scenario with one policy.

We will look at three comparisons:

1. Three predefined policies under 100 sampled scenarios.
2. Five sampled policies under the same 100 scenarios.
3. Three predefined scenarios under the same five sampled policies.

Start with the graphs: what changes, and what stays similar?


### Set up two policy levers

We add two simple interventions that operate from the start of the simulation:

- **Contact reduction:** reduces normal contacts within both regions. It does not change interregional contacts.
- **Shorter infectious period:** reduces the time spent infected in both regions. In this model it speeds up both recovery and death flows; the fatality fraction stays the same.

A lever value of 0 means no change. A value of 0.3 means a 30% reduction from the uncertain baseline value.
These are teaching examples of policy effects.

**Student task:** Which intervention do you think will lower the infection peak most? Make a prediction before running the model.


In [ ]:
from ema_workbench import Policy, Scenario
from ema_workbench.em_framework.samplers import sample_uncertainties, sample_levers
from ema_workbench.analysis import lines


def flu_with_policy(contact_reduction=0, recovery_reduction=0, **parameters):
    parameters["x101"] *= 1 - contact_reduction
    parameters["x102"] *= 1 - contact_reduction
    parameters["x61"] *= 1 - recovery_reduction
    parameters["x62"] *= 1 - recovery_reduction
    return flu_model(**parameters)


policy_model = Model("fluPolicy", function=flu_with_policy)
policy_model.uncertainties = list(model.uncertainties)
policy_model.outcomes = list(model.outcomes)
policy_model.levers = [
    RealParameter("contact_reduction", 0, 0.5),
    RealParameter("recovery_reduction", 0, 0.5),
]

# All ten stocks are recorded. This setting only changes what we plot.
POLICY_OUTCOME = "infected_population_region_1"


### 3.1 Predefined policies: compare with no intervention

We sample 100 scenarios once and reuse them for every policy.
The **base ensemble** is the set of runs with no intervention.

We compare it with two separate interventions: 30% fewer contacts, or a 30% shorter infectious period.
EMA Workbench runs each policy under every scenario: 100 × 3 = **300 simulations**.


In [ ]:
np.random.seed(SEED)
base_scenarios = list(sample_uncertainties(policy_model, 100))

individual_policies = [
    Policy("No intervention", contact_reduction=0, recovery_reduction=0),
    Policy("Reduce contacts", contact_reduction=0.3, recovery_reduction=0),
    Policy("Shorter infectious period", contact_reduction=0, recovery_reduction=0.3),
]

with SequentialEvaluator(policy_model) as evaluator:
    policy_experiments, policy_outcomes = evaluator.perform_experiments(
        scenarios=base_scenarios,
        policies=individual_policies,
    )

print("Number of simulations:", len(policy_experiments))


### Compare the trajectories

Each line is one simulation. Each colour represents a policy.
We show no intervention alongside each intervention in turn.


In [ ]:
for policy_name in ["Reduce contacts", "Shorter infectious period"]:
    fig, axes = lines(
        policy_experiments,
        policy_outcomes,
        outcomes_to_show=POLICY_OUTCOME,
        group_by="policy",
        grouping_specifiers=["No intervention", policy_name],
        titles={POLICY_OUTCOME: POLICY_OUTCOME.replace("_", " ").capitalize()},
    )
    fig.set_size_inches(10, 5)
    ax = axes[POLICY_OUTCOME]
    for line in ax.lines:
        line.set_alpha(0.3)
        line.set_linewidth(0.8)
    ax.set(xlabel="Model time", ylabel="Number of people")
    plt.show()


**Student task:**

- Are the infection peaks generally higher or lower with intervention?
- Do the peaks seem to happen earlier or later?
- Change POLICY_OUTCOME to "deceased_population_region_1" in the setup cell. Rerun that cell and the plotting cell, without rerunning the experiments. Does the intervention also appear to reduce final deaths?

The ensembles can overlap even when a policy helps: a difficult scenario with intervention may still have more infections than an easier scenario without intervention.


### 3.2 Sampled policies: try combinations of both interventions

Instead of choosing the policies ourselves, we ask EMA Workbench to sample five combinations of the two levers.
Each lever can range from 0 to 0.5. Both interventions can now be active together.

We reuse our 100 scenarios: 100 × 5 = **500 additional simulations**.

**Student task:** Look at the policy settings in the table. Which policy would you expect to give the lowest infection peaks?


In [ ]:
np.random.seed(SEED)
sampled_policies = list(sample_levers(policy_model, 5))

# Simple names make it easier to connect the table and graph.
for number, policy in enumerate(sampled_policies, start=1):
    policy.name = f"Policy {number}"

with SequentialEvaluator(policy_model) as evaluator:
    sampled_experiments, sampled_outcomes = evaluator.perform_experiments(
        scenarios=base_scenarios,
        policies=sampled_policies,
    )

print("Number of simulations:", len(sampled_experiments))
display(
    sampled_experiments.groupby("policy", observed=True)[
        ["contact_reduction", "recovery_reduction"]
    ].first()
)


In [ ]:
fig, axes = lines(
    sampled_experiments,
    sampled_outcomes,
    outcomes_to_show=POLICY_OUTCOME,
    group_by="policy",
    titles={POLICY_OUTCOME: POLICY_OUTCOME.replace("_", " ").capitalize()},
)
fig.set_size_inches(10, 5)
ax = axes[POLICY_OUTCOME]
for line in ax.lines:
    line.set_alpha(0.3)
    line.set_linewidth(0.8)
ax.set(xlabel="Model time", ylabel="Number of people")
plt.show()


**Student task:**

- Does the graph match your prediction from the policy table?
- Pick two policies. Which appears to give lower infection peaks?
- Would you need any other information, such as policy costs, before choosing one?

To return to infections after looking at deaths, set POLICY_OUTCOME back to "infected_population_region_1" and rerun the plotting cell. No simulations need to be repeated.


### 3.3 Predefined scenarios: try three named futures

We can also choose the scenarios ourselves, just as we chose the policies in task 3.1.
Here we define three illustrative futures: **Lower transmission**, **Medium transmission**, and **Higher transmission**.

Only the infection rates per contact differ. All other uncertainties are fixed at the midpoints of their ranges.
These are simple examples, not forecasts or estimates of how likely each future is.

We reuse the same five sampled policies: 3 × 5 = **15 additional simulations**.
This mirrors task 3.1: instead of grouping many scenarios by policy, we group several policies by scenario.

**Student task:** Which scenario do you expect to produce the highest infection peaks?


In [ ]:
# Start with the midpoint of every uncertainty range.
scenario_values = {}
for name, label, lower, upper in PARAMETERS:
    scenario_values[name] = (lower + upper) / 2

lower_transmission = scenario_values.copy()
lower_transmission["x91"] = 0.02
lower_transmission["x92"] = 0.02

medium_transmission = scenario_values.copy()
medium_transmission["x91"] = 0.05
medium_transmission["x92"] = 0.05

higher_transmission = scenario_values.copy()
higher_transmission["x91"] = 0.08
higher_transmission["x92"] = 0.08

named_scenarios = [
    Scenario("Lower transmission", **lower_transmission),
    Scenario("Medium transmission", **medium_transmission),
    Scenario("Higher transmission", **higher_transmission),
]

with SequentialEvaluator(policy_model) as evaluator:
    scenario_experiments, scenario_outcomes = evaluator.perform_experiments(
        scenarios=named_scenarios,
        policies=sampled_policies,
    )

print("Number of simulations:", len(scenario_experiments))


### Compare the scenarios

Each colour now represents a scenario. Each line within that colour represents a different policy.
A wider spread within a colour means the five policies produce more different trajectories in that scenario.


In [ ]:
fig, axes = lines(
    scenario_experiments,
    scenario_outcomes,
    outcomes_to_show=POLICY_OUTCOME,
    group_by="scenario",
    titles={POLICY_OUTCOME: POLICY_OUTCOME.replace("_", " ").capitalize()},
)
fig.set_size_inches(10, 5)
axes[POLICY_OUTCOME].set(xlabel="Model time", ylabel="Number of people")
plt.show()


**Student task:**

- Do all three scenarios show a clear outbreak? Which has the highest infection peaks?
- In which scenario do the five policies look most different?
- Change both infection rates in the Higher transmission scenario from 0.08 to 0.1. Rerun the scenario-definition cell and its plot. What changes?

In task 3.1, each colour represented a policy under different external conditions.
Here, each colour represents external conditions under different policies.


## References and interpretation
- [EMA Workbench open exploration](https://emaworkbench.readthedocs.io/en/latest/indepth_tutorial/open-exploration.html)
- [SALib sampling](https://salib.readthedocs.io/en/latest/api/SALib.sample.html): structured Sobol designs.
- [SALib analysis](https://salib.readthedocs.io/en/latest/api/SALib.analyze.html): Sobol and PAWN definitions.
- [Scikit-learn tree rules](https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_text.html).

Do not infer real-world probabilities from ensemble frequencies without justifying the input distributions. Scenario discovery here uses CART-style trees, not PRIM. A low held-out score is a substantive result: simple input rules may not describe the event well.
